# Lab2: 使用 AlexNet 训练 CIFAR-10

实验目标：使用 AlexNet 训练 CIFAR-10

本实验将完成一个完整的图像分类流程，包括：
- 数据加载与预处理
- 模型构建（AlexNet）
- 模型训练
- 模型测试与评估

我们使用 CIFAR-10 数据集。

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import os

# 设置随机种子以确保可重复性
torch.manual_seed(42)

# 定义设备为 GPU（如果可用），否则使用 CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cuda


## 数据预处理

为了提升模型的泛化能力，我们对训练数据进行增强：
- 随机裁剪（RandomCrop）
- 随机翻转（HorizontalFlip）

测试集不做增强，只做标准化处理。

In [2]:
transform_train = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

transform_test = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
])

## 加载数据集

解压数据集，并查看数据集内容

In [3]:
import os
import zipfile
import torchvision
from torch.utils.data import DataLoader

# 本地压缩包路径（优先使用本地）
zip_path = './CIFAR10.zip'
data_root = './CIFAR10'

if not os.path.exists(zip_path):
    print('未找到本地 CIFAR10.zip，尝试联网下载...')
    try:
        # 备用方案：直接使用 torchvision 的在线下载
        trainset = torchvision.datasets.CIFAR10(
            root='./data',
            train=True,
            download=True,
            transform=transform_train,
        )
        testset = torchvision.datasets.CIFAR10(
            root='./data',
            train=False,
            download=True,
            transform=transform_test,
        )

        trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)
        testloader = DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

        eval_dir = 'torchvision CIFAR-10 (online)'
        print('联网下载成功，使用 torchvision 官方数据集。')
    except Exception as e:
        raise RuntimeError(f'联网下载失败，请检查网络或提供本地 {zip_path}: {e}') from e

else:
    print('检测到本地 CIFAR10.zip，使用本地数据。')
    extract_flag = os.path.join(data_root, 'train')

    if not os.path.exists(data_root):
        os.makedirs(data_root)

    if not os.path.exists(extract_flag):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(data_root)
        print('解压完成')
    else:
        print('数据已存在，跳过解压')

    train_dir = os.path.join(data_root, 'train')
    val_dir = os.path.join(data_root, 'val')
    test_dir = os.path.join(data_root, 'test')

    if not os.path.isdir(train_dir):
        raise FileNotFoundError(f'未找到训练集目录: {train_dir}')

    # ImageFolder 需要目录结构为 split/class_x/xxx.jpg
    # 若 test 不是按类别分目录，则回退到 val
    def has_class_subdirs(folder_path):
        if not os.path.isdir(folder_path):
            return False
        for name in os.listdir(folder_path):
            if os.path.isdir(os.path.join(folder_path, name)):
                return True
        return False

    if has_class_subdirs(test_dir):
        eval_dir = test_dir
    elif has_class_subdirs(val_dir):
        eval_dir = val_dir
    else:
        raise FileNotFoundError(
            f'test/val 都不符合 ImageFolder 目录结构，请检查: {test_dir} 和 {val_dir}'
        )

    # 用 ImageFolder 构建数据集，再交给 DataLoader
    trainset = torchvision.datasets.ImageFolder(root=train_dir, transform=transform_train)
    trainloader = DataLoader(trainset, batch_size=128, shuffle=True, num_workers=2)

    testset = torchvision.datasets.ImageFolder(root=eval_dir, transform=transform_test)
    testloader = DataLoader(testset, batch_size=100, shuffle=False, num_workers=2)

    print('使用本地数据目录:', data_root)
    print('评估集目录:', eval_dir)

print('训练样本数:', len(trainset), '评估样本数:', len(testset))

未找到本地 CIFAR10.zip，尝试联网下载...


100.0%
c:\Users\zyh07\Code\Project\AI_Programming\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


联网下载成功，使用 torchvision 官方数据集。
训练样本数: 50000 评估样本数: 10000


## 模型构建：AlexNet

AlexNet 是经典的卷积神经网络，包含：
- 多层卷积层（提取特征）
- 池化层（降维）
- 全连接层（分类）


In [4]:
# 定义 AlexNet 模型
class AlexNet(nn.Module):
    def __init__(self, num_classes=10):
        super(AlexNet, self).__init__()
        # 卷积特征提取
        self.features = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True), # 最大池化
            nn.MaxPool2d(kernel_size=2),
            
            # TODO: 这一层输入是 64，输出是 192，卷积核 3×3，padding=1
            nn.Conv2d(64, 192, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),

            # TODO: 2×2 最大池化，用于下采样
            nn.MaxPool2d(kernel_size=2),

            nn.Conv2d(192, 384, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )
        # 平均池化
        self.avgpool = nn.AdaptiveAvgPool2d((6, 6))
        # 全连接分类
        self.classifier = nn.Sequential(
            nn.Dropout(), # 防止过拟合
            nn.Linear(256 * 6 * 6, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Linear(4096, num_classes),
        )

    def forward(self, x):
        # 数据流
        x = self.features(x)
        # TODO: 自适应平均池化
        x = self.avgpool(x)
        x = torch.flatten(x, 1) # 展平
        # TODO: 全连接分类
        x = self.classifier(x)
        return x

## 初始化模型与优化器
- 损失函数：交叉熵（分类问题常用）
- 优化器：SGD

In [5]:
# 初始化模型、损失函数和优化器
net = AlexNet(num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(net.parameters(), lr=0.001, momentum=0.9, weight_decay=5e-4)

## 训练函数

训练流程包括：
1. 前向传播（forward）
2. 计算损失（loss）
3. 反向传播（backward）
4. 参数更新（optimizer.step）

In [6]:
def train(epoch):
    net.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(device), targets.to(device)
        
        # TODO: 前向传播（输入模型得到输出）
        outputs = net(inputs)
        # TODO:计算损失函数
        loss = criterion(outputs, targets)
        # TODO: 反向传播 + 参数更新（三步）
        optimizer.zero_grad()         # 清空梯度
        loss.backward()               # 计算梯度
        optimizer.step()              # 更新参数
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
    
    print(f"Epoch {epoch+1} - Train Loss: {running_loss/len(trainloader):.4f}, Accuracy: {100.*correct/total:.2f}%")

## 测试函数

测试阶段：
- 不进行梯度计算（节省资源）
- 只评估模型性能

In [ ]:
def test(epoch):
    global best_acc
    net.eval()
    test_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(testloader):
            inputs, targets = inputs.to(device), targets.to(device)
            
            outputs = net(inputs)
            loss = criterion(outputs, targets)
            
            test_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    acc = 100.*correct/total
    print(f"Epoch {epoch+1} - Test Loss: {test_loss/len(testloader):.4f}, Accuracy: {acc:.2f}%")

    if acc > best_acc: # type: ignore
        print('Saving..')
        state = {
            'net': net.state_dict(),
            'acc': acc,
            'epoch': epoch,
        }
        if not os.path.exists('./checkpoint'):
            os.makedirs('./checkpoint')

        torch.save(state, './checkpoint/ckpt.pth')
        best_acc = acc

## 开始训练

设置训练轮数（epoch），循环训练并测试模型。

In [9]:
best_acc = 0
num_epochs = 50

for epoch in range(num_epochs):
    train(epoch)
    test(epoch)

print("Training complete.")

Epoch 1 - Train Loss: 1.5939, Accuracy: 39.84%
Epoch 1 - Test Loss: 1.5104, Accuracy: 42.81%
Saving..
Epoch 2 - Train Loss: 1.5305, Accuracy: 42.17%
Epoch 2 - Test Loss: 1.4576, Accuracy: 45.31%
Saving..
Epoch 3 - Train Loss: 1.4856, Accuracy: 44.30%
Epoch 3 - Test Loss: 1.4441, Accuracy: 45.86%
Saving..
Epoch 4 - Train Loss: 1.4452, Accuracy: 46.15%
Epoch 4 - Test Loss: 1.3503, Accuracy: 49.46%
Saving..
Epoch 5 - Train Loss: 1.4010, Accuracy: 47.79%
Epoch 5 - Test Loss: 1.3425, Accuracy: 50.39%
Saving..
Epoch 6 - Train Loss: 1.3532, Accuracy: 49.84%
Epoch 6 - Test Loss: 1.3107, Accuracy: 52.60%
Saving..
Epoch 7 - Train Loss: 1.3091, Accuracy: 51.70%
Epoch 7 - Test Loss: 1.2952, Accuracy: 53.37%
Saving..
Epoch 8 - Train Loss: 1.2628, Accuracy: 53.53%
Epoch 8 - Test Loss: 1.2053, Accuracy: 55.88%
Saving..
Epoch 9 - Train Loss: 1.2285, Accuracy: 54.81%
Epoch 9 - Test Loss: 1.1360, Accuracy: 58.82%
Saving..
Epoch 10 - Train Loss: 1.1829, Accuracy: 56.67%
Epoch 10 - Test Loss: 1.1481, Accu

# 补充信息
## 网络描述
当前实现的 AlexNet 结构如下：
1. 卷积层1：输入通道数 3，输出通道数 64，卷积核大小 3x3，步长 2，填充 1
2. 激活函数：ReLU
3. 池化层1：最大池化，窗口大小 2x2
4. 卷积层2：输入通道数 64，输出通道数 192，卷积核大小 3x3，步长 1，填充 1
5. 激活函数：ReLU
6. 池化层2：最大池化，窗口大小 2x2
7. 卷积层3：输入通道数 192，输出通道数 384，卷积核大小 3x3，步长 1，填充 1
8. 激活函数：ReLU
9. 卷积层4：输入通道数 384，输出通道数 256，卷积核大小 3x3，步长 1，填充 1
10. 激活函数：ReLU
11. 卷积层5：输入通道数 256，输出通道数 256，卷积核大小 3x3，步长 1，填充 1
12. 激活函数：ReLU
13. 池化层3：最大池化，窗口大小 2x2
14. 自适应平均池化：输出尺寸 6x6
15. 全连接层1：输入特征数 256*6*6，输出特征数 4096（含 Dropout）
16. 激活函数：ReLU
17. 全连接层2：输入特征数 4096，输出特征数 4096（含 Dropout）
18. 激活函数：ReLU
19. 全连接层3：输入特征数 4096，输出特征数 10

**总参数量：61,104,042**

## 参数设置
- 学习率：0.001
- 动量：0.9
- 权重衰减：5e-4
- 批量大小：训练集 128，评估集 100
- 训练轮数：50
- 优化器：SGD
- 损失函数：交叉熵损失（CrossEntropyLoss）
## 运行结果
- 数据规模：训练样本数 50000，评估样本数 10000
- 第 1 轮：Train Loss = 1.5939，Train Acc = 39.84%；Test Loss = 1.5104，Test Acc = 42.81%
- 最佳结果：第 49 轮 Test Acc = 78.23%，Test Loss = 0.6459
- 最后一轮（第 50 轮）：Train Loss = 0.5772，Train Acc = 79.61%；Test Loss = 0.6525，Test Acc = 78.07%
### 结果总结
1. 模型在 50 轮内从 42.81% 提升到 78.07%，总体收敛稳定。
2. 最佳精度出现在第 49 轮（78.23%），第 50 轮略有波动，属常见现象。
3. 训练集与测试集精度接近（79.61% vs 78.07%），暂未出现明显过拟合。